# OpsPilot Ticket Intelligence — TensorFlow Colab Training

This notebook trains TensorFlow/Keras text classifiers for OpsPilot Ticket Intelligence using the fixed English-only train/validation/test splits. It keeps the existing TF-IDF + Logistic Regression models as the benchmark and adds deep learning experiments for category, priority, and optional weak policy-risk labels.

Important leakage rule: `answer` / `reference_answer` is never used as an input feature. It is post-resolution information and is kept only for future response evaluation.

## Section 1 — Runtime/GPU Check

In [ ]:
import sys
import tensorflow as tf

print("Python:", sys.version)
print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU devices:", gpus)
if not gpus:
    print("WARNING: No GPU detected. Training will still run, but it may be slow. In Colab, use Runtime > Change runtime type > GPU.")

## Section 2 — Repo Setup

Option A clones the repo. Replace the placeholder URL if needed. Option B assumes you uploaded or mounted the repo folder yourself.

In [ ]:
# Option A: clone from GitHub
%cd /content
!git clone -b codex/ticket-intelligence https://github.com/shubhamjoshipromail-svg/opspilot-ai.git
%cd /content/opspilot-ai

from pathlib import Path
REPO_ROOT = Path.cwd()
VERTICAL_ROOT = REPO_ROOT / "verticals" / "ticket-intelligence"
print("REPO_ROOT:", REPO_ROOT)
print("VERTICAL_ROOT:", VERTICAL_ROOT)

# If you already uploaded the repo folder, comment out the clone commands above and instead:
# %cd /content/opspilot-ai
# REPO_ROOT = Path.cwd()
# VERTICAL_ROOT = REPO_ROOT / "verticals" / "ticket-intelligence"

In [ ]:
!pip install -q tensorflow pandas numpy scikit-learn matplotlib datasets joblib

## Section 3 — Dataset Build

This regenerates the English-only dataset from `Tobi-Bueck/customer-support-tickets`.

Feature design:
- `customer_message = subject + "\n\n" + body`
- `combined_tags = tag_1 ... tag_8` joined with commas
- `model_text = subject + "\n\n" + body + "\n\nType: " + type + "\nTags: " + combined_tags`
- `true_category = queue`
- `true_priority = priority`
- `reference_answer = answer`

`answer` is excluded from `model_text` to prevent leakage.

In [ ]:
%cd {REPO_ROOT}
!python scripts/build_ticket_dataset.py

In [ ]:
import pandas as pd
from pathlib import Path

normalized_path = REPO_ROOT / "data" / "processed" / "tickets_en_normalized.csv"
if not normalized_path.exists():
    raise FileNotFoundError(f"Missing {normalized_path}. Run the dataset build cell above from REPO_ROOT first.")
normalized = pd.read_csv(normalized_path)
print(normalized.shape)
print(normalized.columns.tolist())
normalized[["subject", "body", "type", "combined_tags", "model_text", "true_category", "true_priority", "reference_answer"]].head(2)

## Section 4 — Create Splits

The split script uses fixed `random_state=42`, keeps train/validation/test separate, and stratifies by category where possible.

In [ ]:
%cd {VERTICAL_ROOT}
!python ml/ticket_intelligence/create_splits.py --input ../../data/processed/tickets_en_normalized.csv

In [ ]:
train = pd.read_csv(VERTICAL_ROOT / "data" / "processed" / "train.csv")
val = pd.read_csv(VERTICAL_ROOT / "data" / "processed" / "val.csv")
test = pd.read_csv(VERTICAL_ROOT / "data" / "processed" / "test.csv")
print(train.shape, val.shape, test.shape)
print(train[["model_text", "customer_message", "subject", "body", "type", "combined_tags", "category", "priority", "reference_answer", "ticket_id"]].head(1).T)

## Section 5 — TensorFlow Category Model

Input: `model_text`

Label: `category` / `true_category`

Leakage prevention: `TextVectorization.adapt()` runs only on training text. Validation is only used for early stopping. Test is only used once for final evaluation.

In [ ]:
!python ml/ticket_intelligence/train_tensorflow_category.py --epochs 10 --batch-size 64

In [ ]:
import json
from pathlib import Path

category_metrics = json.loads(Path("ml/ticket_intelligence/outputs/tensorflow_category_metrics.json").read_text())
{k: category_metrics[k] for k in ["accuracy", "macro_f1", "weighted_f1"]}

## Section 6 — TensorFlow Priority Model

Input: `model_text`

Label: `priority` / `true_priority`

Priority labels are supervised labels from the public dataset, but they may still be noisy and context-dependent.

In [ ]:
!python ml/ticket_intelligence/train_tensorflow_priority.py --epochs 10 --batch-size 64

## Section 7 — Weak Risk Label Creation

These are weak policy-derived labels, not real observed escalation outcomes. They are useful for experimentation and routing-risk analysis, but should not be overstated.

In [ ]:
!python ml/ticket_intelligence/create_risk_labels.py

In [ ]:
risk_summary = json.loads(Path("ml/ticket_intelligence/outputs/risk_label_summary.json").read_text())
risk_summary

## Section 8 — Optional TensorFlow Risk Model

This model predicts `high_risk_policy_label`, a weak policy-derived label. It does not predict true escalation because the dataset does not contain observed escalation outcomes.

In [ ]:
# Optional. Run only after reviewing risk_label_summary.json.
!python ml/ticket_intelligence/train_tensorflow_risk.py --epochs 10 --batch-size 64

## Section 9 — Model Comparison

`model_comparison.json` compares TF-IDF Logistic Regression with TensorFlow models when their metrics are available.

In [ ]:
comparison = json.loads(Path("ml/ticket_intelligence/outputs/model_comparison.json").read_text())
comparison

## Section 10 — Export Results

This creates `tensorflow_training_results.zip` with TensorFlow model artifacts, label mappings, metrics, reports, confusion matrices, error analyses, risk-label audit files, model comparison, and the ML report.

In [ ]:
!zip -r tensorflow_training_results.zip \
  ml/ticket_intelligence/artifacts/tensorflow_* \
  ml/ticket_intelligence/outputs/tensorflow_* \
  ml/ticket_intelligence/outputs/risk_label_* \
  ml/ticket_intelligence/outputs/model_comparison.json \
  docs/ML_RESULTS_REPORT.md

In [ ]:
# Optional Google Drive export
# from google.colab import drive
# drive.mount('/content/drive')
# !cp tensorflow_training_results.zip /content/drive/MyDrive/tensorflow_training_results.zip

## Section 11 — Documentation

After the notebook finishes, review:

- `docs/ML_RESULTS_REPORT.md`
- `ml/ticket_intelligence/outputs/model_comparison.json`
- `ml/ticket_intelligence/outputs/tensorflow_category_metrics.json`
- `ml/ticket_intelligence/outputs/tensorflow_priority_metrics.json`
- `ml/ticket_intelligence/outputs/risk_label_summary.json`

Treat TensorFlow results as an experiment beside the TF-IDF baseline, not a replacement until the metrics and error analysis justify it.